In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import commentjson
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
    AutoModelForTokenClassification
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report
from seqeval.metrics import classification_report as seqeval_report
from tqdm.auto import tqdm
from torchcrf import CRF
from sklearn.model_selection import train_test_split
from safetensors.torch import save_file
# 시드 고정
def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True

set_seed(42)

# 장치 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

# 한국어 특화 모델과 토크나이저 설정
MODEL_NAME = "klue/roberta-base"  # 한국어 특화 모델
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

with open('integrated_data.jsonc', 'r', encoding='utf-8') as f:
        example_data = commentjson.load(f)

with open('integrated_data.jsonc', 'r', encoding='utf-8') as f:
    all_data = commentjson.load(f)




# NER 태그 설정
ner_tags = ["O", "B-BOOK", "I-BOOK", "B-AUTHOR", "I-AUTHOR", "B-FORMAT", "I-FORMAT", "B-DATE", "I-DATE"]
tag2id = {tag: idx for idx, tag in enumerate(ner_tags)}
id2tag = {idx: tag for idx, tag in enumerate(ner_tags)}





# Intent 라벨 설정  
intent_labels = ["기타", "도서검색", "작가검색", "대출베스트", "신간추천", "봇소개", "회원대출", "휴관일", "프로그램", "장소", "도서예약", "희망도서"]
intent2id = {intent: idx for idx, intent in enumerate(intent_labels)}
id2intent = {idx: intent for idx, intent in enumerate(intent_labels)}

def align_entity_with_tokens(text, entities, tokenizer):
    """
    원본 텍스트와 엔티티 위치를 토큰화와 일치시키는 함수
    """
    aligned_entities = []
    
    for entity in entities:
        entity_text = text[entity["start"]:entity["end"]]
        entity_tokens = tokenizer.tokenize(entity_text)
        
        # 전체 텍스트의 토큰화 결과에서 엔티티 시작 위치 찾기
        prefix_text = text[:entity["start"]]
        prefix_tokens = tokenizer.tokenize(prefix_text)
        
        token_start = len(prefix_tokens)
        token_end = token_start + len(entity_tokens) - 1
        
        aligned_entities.append({
            "type": entity["type"],
            "token_start": token_start,
            "token_end": token_end,
            "start": entity["start"],
            "end": entity["end"]
        })
    
    return aligned_entities


# 데이터셋 클래스 정의 (한국어 특화)
class KoreanLibraryDataset(Dataset):
    def __init__(self, data, tokenizer, max_len, tag2id, intent2id):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.tag2id = tag2id
        self.intent2id = intent2id
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        intent = item["intent"]
        
        
        entities = item["entities"]
        
        # 토큰화 전에 원본 텍스트와 엔티티 위치 정렬
        aligned_entities = align_entity_with_tokens(text, entities, self.tokenizer)
        
        # 토큰화 (special_tokens 포함)
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        input_ids = encoding["input_ids"].squeeze()
        attention_mask = encoding["attention_mask"].squeeze()
        
        # NER 레이블 초기화 (모두 O 태그)
        labels = torch.ones(self.max_len, dtype=torch.long) * self.tag2id["O"]
        
        # CLS 토큰 위치 계산
        cls_token_id = self.tokenizer.cls_token_id
        sep_token_id = self.tokenizer.sep_token_id
        
        # CLS 토큰 위치 찾기 (일반적으로 0)
        cls_position = 0
        
        # 각 엔티티에 대해 BIO 태그 적용 (special token 고려)
        for entity in aligned_entities:
            entity_type = entity["type"]
            # CLS 토큰 때문에 오프셋 +1
            token_start = entity["token_start"] + 1
            token_end = entity["token_end"] + 1
            
            # 시작 토큰에 B- 태그 적용
            if token_start < self.max_len:
                labels[token_start] = self.tag2id[f"B-{entity_type}"]
            
            # 나머지 토큰에 I- 태그 적용
            for i in range(token_start + 1, token_end + 1):
                if i < self.max_len:
                    labels[i] = self.tag2id[f"I-{entity_type}"]
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "intent_label": torch.tensor(intent, dtype=torch.long),
            "ner_labels": labels
        }


intent_list = [d["intent"] for d in all_data]
train_data, valid_data = train_test_split(
    all_data,
    test_size=0.2,
    random_state=42,
    stratify=intent_list
)

train_dataset = KoreanLibraryDataset(train_data, tokenizer, MAX_LENGTH, tag2id, intent2id)
valid_dataset = KoreanLibraryDataset(valid_data, tokenizer, MAX_LENGTH, tag2id, intent2id)

# 데이터로더 생성
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)

# 통합 모델 정의 (Intent 분류와 NER을 동시에 수행)
class JointIntentNERModel(nn.Module):
    def __init__(self, model_name, num_intents, num_ner_tags, dropout_prob=0.1):
        super(JointIntentNERModel, self).__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        
        # Intent 분류를 위한 레이어 (기존과 동일)
        self.intent_classifier = nn.Sequential(
            nn.Linear(self.roberta.config.hidden_size, self.roberta.config.hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(self.roberta.config.hidden_size, num_intents)
        )
        
        # NER을 위한 레이어 (마지막 Linear 레이어만 필요)
        # CRF에 입력으로 넣을 각 태그에 대한 점수(emission score)를 계산합니다.
        self.ner_emitter = nn.Sequential(
            nn.Linear(self.roberta.config.hidden_size, self.roberta.config.hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(self.roberta.config.hidden_size // 2, num_ner_tags)
        )
        
        # --- [CRF 추가] ---
        # CRF 레이어 초기화. num_tags는 NER 태그의 개수입니다.
        self.crf = CRF(num_ner_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, ner_labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        
        # Intent 분류 (기존과 동일)
        pooled_output = sequence_output[:, 0, :]
        intent_logits = self.intent_classifier(pooled_output)
        
        # --- [NER 로직 수정] ---
        # 1. Emission score 계산
        ner_emissions = self.ner_emitter(sequence_output)
        
        # 2. 훈련/추론 분기
        if ner_labels is not None:
            # 훈련 시: CRF 레이어를 이용해 손실(negative log-likelihood)을 계산합니다.
            # mask는 패딩된 부분의 손실을 계산하지 않도록 합니다.
            # CRF 라이브러리는 boolean 마스크를 기대할 수 있으므로 .bool() 또는 .byte()를 사용합니다.
            mask = attention_mask.bool()
            ner_loss = -self.crf(ner_emissions, ner_labels, mask=mask, reduction='mean')
            return intent_logits, ner_loss # ner_loss를 직접 반환
        else:
            # 추론 시: Viterbi 알고리즘을 사용해 최적의 태그 시퀀스를 예측(디코딩)합니다.
            mask = attention_mask.bool()
            ner_preds = self.crf.decode(ner_emissions, mask=mask)
            return intent_logits, ner_preds # 예측된 태그 시퀀스를 반환

# 모델 초기화
model = JointIntentNERModel(
    model_name=MODEL_NAME, 
    num_intents=len(intent_labels), 
    num_ner_tags=len(ner_tags)
)
model.to(device)

# 옵티마이저 및 스케줄러 설정
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_dataloader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=total_steps
)

# 손실 함수
intent_criterion = nn.CrossEntropyLoss()
ner_criterion = nn.CrossEntropyLoss(ignore_index=-100)  # padding된 부분은 손실 계산에서 제외



# 모델 저장 함수
def save_model(model, tokenizer, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 모델 가중치를 safetensors 형식으로 저장
    model_weights = model.state_dict()
    save_file(model_weights, os.path.join(output_dir, "model.safetensors"))
    
    # 토크나이저 저장
    tokenizer.save_pretrained(output_dir)
    
    # 모델 설정 저장
    config = {
        "intent_labels": intent_labels,
        "ner_tags": ner_tags,
        "max_length": MAX_LENGTH
    }
    with open(os.path.join(output_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

def train_and_evaluate(model, train_dataloader, valid_dataloader, optimizer, scheduler, 
                      device, num_epochs, output_dir, patience=3):
    best_f1 = 0

    epochs_no_improve = 0

    


    main_progress_bar = tqdm(total=num_epochs * len(train_dataloader) + num_epochs * len(valid_dataloader), desc="훈련 진행률")

    for epoch in range(num_epochs):
        main_progress_bar.set_description(f"에포크 {epoch+1}/{num_epochs}")
        
        # --- 훈련 단계 ---
        model.train()
        total_intent_loss, total_ner_loss = 0, 0
        
        for batch in train_dataloader:
            # ... (기존 훈련 코드 그대로)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            intent_label = batch["intent_label"].to(device)
            ner_labels = batch["ner_labels"].to(device)
            
            intent_logits, ner_loss = model(input_ids, attention_mask, ner_labels=ner_labels)
            intent_loss = intent_criterion(intent_logits, intent_label)
            loss = intent_loss + ner_loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_intent_loss += intent_loss.item()
            total_ner_loss += ner_loss.item()
            main_progress_bar.update(1)

        # --- 평가 단계 ---
        model.eval()
        total_intent_loss_val = 0
        intent_preds_list, intent_labels_list = [], []
        ner_preds_list, ner_true_list = [], []
        
        with torch.no_grad():
            for batch in valid_dataloader:
                # ... (기존 평가 코드 그대로)
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                intent_label = batch["intent_label"].to(device)
                ner_labels = batch["ner_labels"].to(device)
                
                intent_logits, ner_preds_decoded = model(input_ids, attention_mask)
                intent_loss_val = intent_criterion(intent_logits, intent_label)
                total_intent_loss_val += intent_loss_val.item()
                
                intent_preds_list.extend(torch.argmax(intent_logits, dim=1).cpu().numpy())
                intent_labels_list.extend(intent_label.cpu().numpy())
                
                ner_true_labels = ner_labels.cpu().numpy()
                for i, mask in enumerate(attention_mask.cpu().numpy()):
                    pred_tags = [id2tag[p] for p in ner_preds_decoded[i]]
                    true_len = int(mask.sum())
                    true_tags = [id2tag[t] for t in ner_true_labels[i][:true_len]]
                    ner_preds_list.append(pred_tags[1:-1])
                    ner_true_list.append(true_tags[1:-1])

                main_progress_bar.update(1)

        # 평가 지표 계산 (이 부분이 for loop 안에 있어야 함)
        val_intent_accuracy = accuracy_score(intent_labels_list, intent_preds_list)
        val_intent_f1 = f1_score(intent_labels_list, intent_preds_list, average='weighted', zero_division=0)
        ner_report = seqeval_report(ner_true_list, ner_preds_list, output_dict=True, zero_division=0)
        val_ner_f1 = ner_report['weighted avg']['f1-score']
        
        # 각 에포크마다 결과 출력 (이 부분도 for loop 안에 있어야 함)
        avg_train_intent_loss = total_intent_loss / len(train_dataloader)
        avg_train_ner_loss = total_ner_loss / len(train_dataloader)
        avg_val_intent_loss = total_intent_loss_val / len(valid_dataloader)

        print(f"\n에포크 {epoch+1}/{num_epochs}")
        print(f"훈련 - Intent Loss: {avg_train_intent_loss:.4f}, NER Loss: {avg_train_ner_loss:.4f}")
        print(f"검증 - Intent Loss: {avg_val_intent_loss:.4f}")
        print(f"검증 - Intent Accuracy: {val_intent_accuracy:.4f}, Intent F1: {val_intent_f1:.4f}")
        print(f"검증 - NER F1: {val_ner_f1:.4f}")

        # 전체 평균 F1 계산 및 모델 저장 (이 부분도 for loop 안에 있어야 함)
        overall_f1 = (val_intent_f1 + val_ner_f1) / 2
        if overall_f1 > best_f1:
            best_f1 = overall_f1
            print(f"새로운 최고 성능! 평균 F1: {best_f1:.4f} - 모델을 저장합니다.")
            save_model(model, tokenizer, output_dir)
            epochs_no_improve = 0 
        else:
            epochs_no_improve += 1 # 성능이 개선되지 않았으므로 카운터 증가
            print(f"성능 개선 없음. 최고 F1: {best_f1:.4f} (정체: {epochs_no_improve}/{patience})")
        if epochs_no_improve >= patience:
            print(f"\n{patience} 에포크 동안 성능 개선이 없어 훈련을 조기 종료합니다.")
            break
    
    main_progress_bar.close()
    return best_f1

# 훈련 실행
best_f1 = 0
output_dir = "./korean_library_chatbot_model"
logging_steps = 10  # 10배치마다 진행 상황 업데이트

# 훈련 시작
print("\n훈련 시작...")
best_f1 = train_and_evaluate(
    model=model,
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=EPOCHS,
    output_dir=output_dir
)

print(f"\n훈련 완료! 최종 모델이 {output_dir}에 저장되었습니다. (최고 평균 F1: {best_f1:.4f})")

# 실제 텍스트에서 엔티티를 추출하는 함수 개선
def extract_entities_from_tokens(text, token_predictions, offset_mapping, id2tag):
    entities = []
    current_entity = None

    for i, (pred, (start, end)) in enumerate(zip(token_predictions, offset_mapping)):
        # [CLS], [SEP] 같은 특수 토큰은 건너뜁니다.
        if start == end:
            continue

        tag = id2tag[pred]

        # --- 핵심 수정 부분 시작 ---
        # B 태그가 나오거나, I 태그가 나왔지만 이전 개체가 없는 경우 -> 새로운 개체 시작
        if tag.startswith("B-") or (tag.startswith("I-") and current_entity is None):
            # 이전에 처리하던 개체가 있었다면 먼저 저장
            if current_entity is not None:
                entities.append(current_entity)

            entity_type = tag[2:]  # "B-" 또는 "I-" 접두사 제거
            current_entity = {
                "type": entity_type,
                "start": start,
                "end": end,
                "text": text[start:end]
            }
        
        # I 태그가 나왔고, 이전 개체와 타입이 같은 경우 -> 개체 확장
        elif tag.startswith("I-") and current_entity is not None and current_entity["type"] == tag[2:]:
            current_entity["end"] = end
            current_entity["text"] = text[current_entity["start"]:end]

        # O 태그가 나오거나, 다른 타입의 개체가 시작된 경우 -> 현재 개체 저장 후 초기화
        else: # tag == "O" 또는 다른 타입의 B-/I- 태그가 나온 경우
            if current_entity is not None:
                entities.append(current_entity)
            current_entity = None
        # --- 핵심 수정 부분 끝 ---

    # 마지막 토큰까지 처리한 후, 남아있는 개체가 있으면 저장
    if current_entity is not None:
        entities.append(current_entity)

    # 중복 및 포함 관계 처리 (이 부분은 그대로 유지해도 좋습니다)
    if not entities:
        return []

    filtered_entities = []
    # 텍스트 길이를 기준으로 정렬하여 긴 개체를 먼저 처리
    sorted_entities = sorted(entities, key=lambda e: len(e["text"]), reverse=True)
    
    for entity in sorted_entities:
        is_contained = False
        for filtered in filtered_entities:
            # 현재 개체가 이미 추가된 더 긴 개체에 포함되는지 확인
            if (entity["start"] >= filtered["start"] and entity["end"] <= filtered["end"]):
                is_contained = True
                break
        
        if not is_contained:
            filtered_entities.append(entity)

    # 시작 위치(start) 기준으로 최종 정렬
    return sorted(filtered_entities, key=lambda e: e["start"])

# 모델 추론 함수 개선
def predict_intent_and_entities(text, model, tokenizer, id2intent, id2tag):
    model.eval()
    
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_offsets_mapping=True,
        return_tensors="pt"
    )
    
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    offset_mapping = encoding["offset_mapping"].squeeze().cpu().numpy()
    
    with torch.no_grad():
        # [수정] 모델 반환값 변경
        intent_logits, ner_preds_decoded = model(input_ids, attention_mask)
        
        # Intent 예측
        intent_pred = torch.argmax(intent_logits, dim=1).cpu().numpy()[0]
        intent_name = id2intent[intent_pred]
        
        # [수정] NER 예측 결과는 이미 디코딩된 ID 리스트
        ner_pred_ids = ner_preds_decoded[0] # 배치 크기가 1이므로 첫 번째 결과 사용
        
        # 토큰화된 결과 확인 (디버깅용)
        # ... 기존 코드와 동일 ...
        # (단, ner_pred 대신 ner_pred_ids 사용)
        
        # 실제 텍스트 범위에 맞게 엔티티 추출
        # extract_entities_from_tokens 함수는 그대로 사용 가능
        entities = extract_entities_from_tokens(text, ner_pred_ids, offset_mapping, id2tag)
    
    return {
        "intent": intent_name,
        "entities": entities
    }
def postprocess_prediction(text, prediction, common_authors_set=None, common_books_set=None):
    current_intent = prediction["intent"]
    original_model_intent = prediction["intent"] 
    entities = prediction["entities"]
    
    has_author_entity = any(e["type"] == "AUTHOR" for e in entities)
    has_book_entity = any(e["type"] == "BOOK" for e in entities)

    # 시나리오 1: 작가 엔티티만 있고, 모델이 '도서검색'으로 예측한 경우 -> '작가검색'으로 보정
    if original_model_intent == "도서검색" and has_author_entity and not has_book_entity:
        current_intent = "작가검색" 

    # 시나리오 2: 책 엔티티만 있고, 모델이 '작가검색'으로 예측한 경우 -> '도서검색'으로 보정
    elif original_model_intent == "작가검색" and has_book_entity and not has_author_entity:
        current_intent = "도서검색"
        
    if original_model_intent != current_intent:
        print(f"INFO: Intent corrected by postprocessing. Original: '{original_model_intent}', Corrected: '{current_intent}'. Text: '{text}'")

    return {
        "intent": current_intent,
        "entities": entities,
        "original_intent": original_model_intent,
        "text": text  # 함수의 인자로 받은 text를 사용
    }

# 개선된 추론 실행
print("\n개선된 추론 예시:")
sample_texts = [
    "스즈미야 하루히의 우울 도서관에에 있나요?",
    "전생했더니 슬라임이였던 건에 대하여 책 있어?",
    "나루토 책 있냐??",
    "한강 책 있냐??"
]

for sample_text in sample_texts:
    print(f"\n입력 텍스트: {sample_text}")
    
    # 원본 예측
    raw_prediction = predict_intent_and_entities(
        sample_text, model, tokenizer, id2intent, id2tag
    )
    
    # 후처리로 예측 개선
    prediction = postprocess_prediction(sample_text, raw_prediction)
    
    print(f"예측 결과:")
    print(f"  의도: {prediction['intent']}")
    print(f"  개체:")
    if not prediction['entities']:
        print("    - 개체 없음")
    else:
        for entity in prediction['entities']:
            print(f"    - {entity['type']}: '{entity['text']}' (위치: {entity['start']}-{entity['end']})")

사용 장치: cuda


Some weights of RobertaModel were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



훈련 시작...


훈련 진행률:   0%|          | 0/6150 [00:00<?, ?it/s]


에포크 1/50
훈련 - Intent Loss: 2.5169, NER Loss: 31.7582
검증 - Intent Loss: 2.3352
검증 - Intent Accuracy: 0.5000, Intent F1: 0.3333
검증 - NER F1: 0.0000
새로운 최고 성능! 평균 F1: 0.1667 - 모델을 저장합니다.

에포크 2/50
훈련 - Intent Loss: 2.0770, NER Loss: 13.0828
검증 - Intent Loss: 1.8286
검증 - Intent Accuracy: 0.5000, Intent F1: 0.3333
검증 - NER F1: 0.8149
새로운 최고 성능! 평균 F1: 0.5741 - 모델을 저장합니다.

에포크 3/50
훈련 - Intent Loss: 1.7571, NER Loss: 5.3418
검증 - Intent Loss: 1.6987
검증 - Intent Accuracy: 0.5000, Intent F1: 0.3333
검증 - NER F1: 0.8994
새로운 최고 성능! 평균 F1: 0.6164 - 모델을 저장합니다.

에포크 4/50
훈련 - Intent Loss: 1.6823, NER Loss: 2.2734
검증 - Intent Loss: 1.6227
검증 - Intent Accuracy: 0.5000, Intent F1: 0.3333
검증 - NER F1: 0.9454
새로운 최고 성능! 평균 F1: 0.6393 - 모델을 저장합니다.

에포크 5/50
훈련 - Intent Loss: 1.4147, NER Loss: 1.2102
검증 - Intent Loss: 0.9646
검증 - Intent Accuracy: 0.6964, Intent F1: 0.6309
검증 - NER F1: 0.9645
새로운 최고 성능! 평균 F1: 0.7977 - 모델을 저장합니다.

에포크 6/50
훈련 - Intent Loss: 0.7261, NER Loss: 0.5326
검증 - Intent Loss: 0.5630
